Download TinyStories Dataset from huggingface

In [9]:
import sys
!{sys.executable} -m hf auth login

c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\python.exe: No module named hf


In [10]:
import sys
!{sys.executable} -m pip install datasets

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [11]:
from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories")

# or load the separate splits if the dataset has train/validation/test splits
train_dataset = load_dataset("roneneldan/TinyStories", split="train")
valid_dataset = load_dataset("roneneldan/TinyStories", split="validation")
test_dataset  = load_dataset("roneneldan/TinyStories", split="test")

c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MSI-1\.cache\huggingface\hub\datasets--roneneldan--TinyStories. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an

ValueError: Unknown split "test". Should be one of ['train', 'validation'].

In [12]:
train_dataset.shape

(2119719, 1)

In [13]:
import torch

In [15]:
type(train_dataset)

datasets.arrow_dataset.Dataset

In [16]:
for i in train_dataset:
    print(i)
    break


{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [17]:
dataset.shape

{'train': (2119719, 1), 'validation': (21990, 1)}

In [28]:
import sys
!{sys.executable} -m pip install tokenizers




You should consider upgrading via the 'c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [18]:

gsm8k = load_dataset("openai/gsm8k", "main")

c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MSI-1\.cache\huggingface\hub\datasets--openai--gsm8k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 53831.21 examples/s]


In [19]:
gsm8k.shape

{'train': (7473, 2), 'test': (1319, 2)}

In [23]:
gsm8k['train'][0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

In [47]:
def get_training_corpus(tiny_stories_limit=500000):
    """
    Generator that streams formatting training data to the tokenizer.
    Yields formatted GSM8K first, then a subset of TinyStories.
    """
    print("Loading GSM8K...")
    gsm8k = load_dataset("gsm8k", "main", split="train")
    
    # 1. Yield GSM8K formatted for the TRM
    for row in gsm8k:
        question = row["question"]
        raw_answer = row["answer"]
        
        # GSM8K separates the reasoning from the final answer with "####"
        parts = raw_answer.split("####")
        reasoning = parts[0].strip()
        final_answer = parts[1].strip() if len(parts) > 1 else ""
        
        # Format it exactly how the model should see it
        formatted_text = (
            f"[INST] {question} [/INST]\n"
            f"[THINK]\n{reasoning}\n[/THINK]\n"
            f"[ANSWER] {final_answer} [/ANSWER]"
        )
        yield formatted_text

    print("Streaming TinyStories...")
    # 2. Yield TinyStories (Streaming so we don't download it all at once)
    tiny_stories = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
    
    for i, row in enumerate(tiny_stories):
        if i >= tiny_stories_limit:
            break
        yield row["text"]

Train the tokenizer

In [7]:
import os
from datasets import load_dataset
from tokenizers import Tokenizer, Regex
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Sequence, Whitespace, Split
from tokenizers.processors import TemplateProcessing

c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [48]:
def train_hf_tokenizer(output_dir: str = "./trm_tokenizer"):
    print("Initializing BPE Tokenizer...")
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

    # The Math Hack: Split by whitespace, then isolate digits
    tokenizer.pre_tokenizer = Sequence([
        Whitespace(),
        Split(Regex(r"\d"), behavior="isolated") 
    ])

    # Structural Tokens for G-TRM
    special_tokens = [
        "[PAD]", "[UNK]", "[BOS]", "[EOS]", 
        "[INST]", "[/INST]", "[THINK]", "[/THINK]", "[ANSWER]", "[/ANSWER]"
    ]

    # 4096 Vocab Size to keep embedding layer ~1M parameters
    trainer = BpeTrainer(
        vocab_size=4096, 
        special_tokens=special_tokens,
        initial_alphabet=list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!@#$%^&*()_+-=[]{}|;':\",./<>?`~ \n"),
        show_progress=True
    )

    print("Training tokenizer from Hugging Face iterator...")
    # Train using the generator!
    tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)

    # Post-Processing: Auto-wrap sequences in BOS and EOS
    tokenizer.post_processor = TemplateProcessing(
        single="[BOS] $A [EOS]",
        pair="[BOS] $A [EOS] $B:1 [EOS]:1",
        special_tokens=[
            ("[BOS]", tokenizer.token_to_id("[BOS]")),
            ("[EOS]", tokenizer.token_to_id("[EOS]")),
        ],
    )

    # Save to disk
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    save_path = os.path.join(output_dir, "tokenizer.json")
    tokenizer.save(save_path)
    print(f"\nSuccess! Custom Micro-BPE tokenizer saved to {save_path}")

    return tokenizer

In [1]:
my_tokenizer = train_hf_tokenizer()

NameError: name 'train_hf_tokenizer' is not defined

In [2]:
test_string = "[INST] Alice has 14 apples. [/INST]"

In [3]:
enc_str = my_tokenizer.encode(test_string)
enc_str.tokens

NameError: name 'my_tokenizer' is not defined

In [5]:
tokenizer_path = "C:/Users/MSI-1/Desktop/Adnan's Major/TinyRecursiveModels/trm_tokenizer/tokenizer.json"


In [17]:
tokenizer = Tokenizer.from_file(tokenizer_path)


In [18]:
pad_token_id = tokenizer.token_to_id("[PAD]")
print(pad_token_id)


0


In [15]:
text = "[ADL] hi,"
enc = tokenizer.encode(text)
enc.tokens, enc.ids


(['[BOS]', '[', 'A', 'D', 'L', ']', 'hi', ',', '[EOS]'],
 [2, 70, 44, 47, 55, 72, 2824, 23, 3])

In [9]:
prompt = "[INST] Alice has 15 apples. She eats 3. How many are left? [/INST]\n[THINK]"
encoded = tokenizer.encode(prompt)
encoded.tokens

['[BOS]',
 '[INST]',
 'Alice',
 'has',
 '1',
 '5',
 'apples',
 '.',
 'She',
 'eat',
 's',
 '3',
 '.',
 'How',
 'many',
 'are',
 'left',
 '?',
 '[/INST]',
 '[THINK]',
 '[EOS]']

In [10]:
encoded.ids

[2,
 4,
 2134,
 681,
 28,
 32,
 1809,
 25,
 192,
 590,
 94,
 30,
 25,
 1503,
 597,
 254,
 1152,
 42,
 5,
 6,
 3]

In [21]:
gsm8k['train'][0]

NameError: name 'gsm8k' is not defined

Preparing TinyStories & Gsm8k for training

In [1]:
import torch
from datasets import load_dataset
from tokenizers import Tokenizer
from torch.utils.data import DataLoader

# ==========================================
# 1. Configuration & Tokenizer Setup
# ==========================================
MAX_SEQ_LEN = 512 # A safe, efficient context window for a 7M parameter edge model
BATCH_SIZE = 16

print("Loading Tokenizer...")
tokenizer = Tokenizer.from_file("trm_tokenizer/tokenizer_mask_1.json")

# Tell the rust tokenizer to handle truncation and padding automatically!
pad_token_id = tokenizer.token_to_id("[PAD]")
tokenizer.enable_truncation(max_length=MAX_SEQ_LEN)
# tokenizer.enable_padding(pad_id=pad_token_id, length=MAX_SEQ_LEN)



c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Tokenizer...


In [2]:
# ==========================================
# 2. Formatting Functions
# ==========================================
import re

def format_gsm8k(example):
    """Converts raw GSM8K into the TRM Template."""
    question = example["question"]
    parts = example["answer"].split("####")
    raw_reasoning = parts[0].strip()
    clean_reasoning = re.sub(r'<<.*?>>', '', raw_reasoning)
    final_answer = parts[1].strip() if len(parts) > 1 else ""
    
    formatted_text = (
        f"[INST] {question} [/INST]\n"
        f"[THINK]\n{clean_reasoning}\n[/THINK]\n"
        f"[ANSWER] {final_answer} [/ANSWER]"
    )
    return {"text": formatted_text}

def format_tinystories(example):
    """TinyStories needs no formatting, just a 'text' key mapping."""
    return {"text": example["text"]}

In [15]:
def tokenize_and_prepare_labels(examples):
    # Use encode_batch but ensure it doesn't pad here
    encodings = tokenizer.encode_batch(examples["text"])
    
    input_ids = []
    labels = []
    
    for enc in encodings:
        # 1. Grab MAX_SEQ_LEN + 1 so we have room to shift!
        ids = enc.ids[:MAX_SEQ_LEN + 1] 
        
        # 2. THE AR SHIFT
        shifted_inputs = ids[:-1] # Everything except the last token
        shifted_labels = ids[1:]  # Everything except the first token
        
        input_ids.append(shifted_inputs)
        labels.append(shifted_labels) 
        
    return {
        "input_ids": input_ids,
        "labels": labels
    }

In [16]:
# ==========================================
# 4. Pipeline Execution
# ==========================================
print("Loading Datasets...")
# Load GSM8K
gsm8k_dataset = load_dataset("gsm8k", "main", split="train")

# Load a subset of TinyStories (e.g., 100,000 stories so it doesn't overwhelm math)
tinystories_dataset = load_dataset("roneneldan/TinyStories", split="train[:100000]")

print("Formatting Text...")
# Apply the formatting templates
gsm8k_dataset = gsm8k_dataset.map(format_gsm8k, remove_columns=gsm8k_dataset.column_names)
tinystories_dataset = tinystories_dataset.map(format_tinystories, remove_columns=tinystories_dataset.column_names)

print("Tokenizing... (This is very fast)")
# Apply tokenization using batched mapping
gsm8k_tokenized = gsm8k_dataset.map(tokenize_and_prepare_labels, batched=True, batch_size=1000, remove_columns=["text"])
tinystories_tokenized = tinystories_dataset.map(tokenize_and_prepare_labels, batched=True, batch_size=1000, remove_columns=["text"])

# Tell HF datasets to return actual PyTorch Tensors
gsm8k_tokenized.set_format(type="torch", columns=["input_ids", "labels"])
tinystories_tokenized.set_format(type="torch", columns=["input_ids", "labels"])

Loading Datasets...
Formatting Text...
Tokenizing... (This is very fast)


Map: 100%|██████████| 100000/100000 [00:14<00:00, 6759.88 examples/s]


In [17]:
from torch.nn.utils.rnn import pad_sequence

# Make sure this is defined from your tokenizer!
pad_token_id = tokenizer.token_to_id("[PAD]")

def trm_collate_fn(batch):
    """
    Dynamically pads batches to the longest sequence in the batch, saving VRAM.
    """
    # Extract tensors from the batch
    input_ids = [item['input_ids'] for item in batch]
    labels = [item['labels'] for item in batch]

    # pad_sequence automatically stacks them and fills the empty space
    padded_input_ids = pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": padded_input_ids,
        "labels": padded_labels
    }

In [23]:

# ==========================================
# 5. Create PyTorch DataLoaders
# ==========================================
# For curriculum learning, we keep them separate so you can control the mix during training
gsm8k_loader = DataLoader(gsm8k_tokenized, batch_size=BATCH_SIZE, shuffle=True, 
    collate_fn=trm_collate_fn)
tinystories_loader = DataLoader(tinystories_tokenized, batch_size=BATCH_SIZE, shuffle=True, 
    collate_fn=trm_collate_fn)

print("\n--- Pipeline Complete! ---")
print(f"GSM8K Batches available: {len(gsm8k_loader)}")
print(f"TinyStories Batches available: {len(tinystories_loader)}")

# Let's verify a batch!
sample_batch = next(iter(gsm8k_loader))
print(f"\nShape of input_ids: {sample_batch['input_ids'].shape}")
print(f"Shape of labels: {sample_batch['labels'].shape}")
print(f"First 10 input_ids: {sample_batch['input_ids'][0][:10]}")
print(f"Last 10 labels (Notice the -100 padding!): {sample_batch['labels'][0][-10:]}")


--- Pipeline Complete! ---
GSM8K Batches available: 468
TinyStories Batches available: 6250

Shape of input_ids: torch.Size([16, 422])
Shape of labels: torch.Size([16, 422])
First 10 input_ids: tensor([   2,    5, 1131,  456,  383,  636, 2347,  403, 1305,   30])
Last 10 labels (Notice the -100 padding!): tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100])


In [28]:
a = tokenizer.decode(sample_batch['input_ids'][0][:10].tolist(), skip_special_tokens=False)
a

'[BOS] [INST] Sarah is in charge of making 2'

In [29]:
cleaned = torch.where(sample_batch['labels'] == -100, pad_token_id, sample_batch['labels'])
b = tokenizer.decode(cleaned[0][:10].tolist(), skip_special_tokens=False)
b

'[INST] Sarah is in charge of making 2 cop'

In [21]:
print("Inputs: ", sample_batch['input_ids'][0][:10].tolist())
print("Labels: ", sample_batch['labels'][0][:10].tolist())

Inputs:  [2, 5, 761, 1416, 193, 77, 1239, 360, 94, 467]
Labels:  [5, 761, 1416, 193, 77, 1239, 360, 94, 467, 422]


Training Loop

In [30]:
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass
import math
import torch
import copy
import torch.nn.functional as F
from torch import nn
from pydantic import BaseModel
import random
from models.common import trunc_normal_init_
from models.layers import rms_norm, LinearSwish, SwiGLU, Attention, RotaryEmbedding, CosSin, CastedEmbedding, CastedLinear
from models.sparse_embedding import CastedSparseEmbedding

IGNORE_LABEL_ID = -100

@dataclass
class TinyRecursiveReasoningModel_ACTV1InnerCarry:
    z_L: torch.Tensor



@dataclass
class TinyRecursiveReasoningModel_ACTV1Carry:
    inner_carry: TinyRecursiveReasoningModel_ACTV1InnerCarry
    
    steps: torch.Tensor
    halted: torch.Tensor
    
    current_data: Dict[str, torch.Tensor]


class TinyRecursiveReasoningModel_ACTV1Config(BaseModel):
    batch_size: int
    seq_len: int
    puzzle_emb_ndim: int = 0
    num_puzzle_identifiers: int
    vocab_size: int

    H_cycles: int
    L_cycles: int

    H_layers: int # ignored
    L_layers: int

    # Transformer config
    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str

    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    
    # Halting Q-learning config
    halt_max_steps: int
    halt_exploration_prob: float

    forward_dtype: str = "bfloat16"

    # Alexia: added
    mlp_t: bool = False # use mlp on L instead of transformer
    puzzle_emb_len: int = 16 # if non-zero, its specified to this value
    no_ACT_continue: bool =  True # No continue ACT loss, only use the sigmoid of the halt which makes much more sense
    causal: bool = False

class TinyRecursiveReasoningModel_ACTV1Block(nn.Module):
    def __init__(self, config: TinyRecursiveReasoningModel_ACTV1Config) -> None:
        super().__init__()

        self.config = config
        if self.config.mlp_t:
            self.puzzle_emb_len = -(self.config.puzzle_emb_ndim // -self.config.hidden_size) if self.config.puzzle_emb_len == 0 else self.config.puzzle_emb_len
            self.mlp_t = SwiGLU(
                hidden_size=self.config.seq_len + self.puzzle_emb_len, # L
                expansion=config.expansion,
            )
        else:
            self.self_attn = Attention(
                hidden_size=config.hidden_size,
                head_dim=config.hidden_size // config.num_heads,
                num_heads=config.num_heads,
                num_key_value_heads=config.num_heads,
                causal=config.causal
            )
        self.mlp = SwiGLU(
            hidden_size=config.hidden_size,
            expansion=config.expansion,
        )
        self.norm_eps = config.rms_norm_eps

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor) -> torch.Tensor:
        # B, L, D = hidden_states.shape
        # Post Norm
        if self.config.mlp_t:
            hidden_states = hidden_states.transpose(1,2)
            out = self.mlp_t(hidden_states)
            hidden_states = rms_norm(hidden_states + out, variance_epsilon=self.norm_eps)
            hidden_states = hidden_states.transpose(1,2)
        else:
            # Self Attention
            hidden_states = rms_norm(hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states), variance_epsilon=self.norm_eps)
        # Fully Connected
        out = self.mlp(hidden_states)
        hidden_states = rms_norm(hidden_states + out, variance_epsilon=self.norm_eps)
        return hidden_states

class TinyRecursiveReasoningModel_ACTV1ReasoningModule(nn.Module):
    def __init__(self, layers: List[TinyRecursiveReasoningModel_ACTV1Block]):
        super().__init__()
        self.layers = torch.nn.ModuleList(layers)

    def forward(self, hidden_states: torch.Tensor, **kwargs) -> torch.Tensor:
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, **kwargs)
        return hidden_states


class TinyRecursiveReasoningModel_ACTV1_Inner(nn.Module):
    def __init__(self, config: TinyRecursiveReasoningModel_ACTV1Config) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)

        # I/O

        self.embed_scale = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale

        self.embed_tokens = CastedEmbedding(self.config.vocab_size, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.lm_head      = CastedLinear(self.config.hidden_size, self.config.vocab_size, bias=False)
        self.q_head       = CastedLinear(self.config.hidden_size, 2, bias=True)

        self.puzzle_emb_len = -(self.config.puzzle_emb_ndim // -self.config.hidden_size)  if self.config.puzzle_emb_len == 0 else self.config.puzzle_emb_len  # ceil div
        if self.config.puzzle_emb_ndim > 0:
            # Zero init puzzle embeddings
            self.puzzle_emb = CastedSparseEmbedding(self.config.num_puzzle_identifiers, self.config.puzzle_emb_ndim,
                                                    batch_size=self.config.batch_size, init_std=0, cast_to=self.forward_dtype)

        # LM Blocks
        if self.config.pos_encodings == "rope":
            self.rotary_emb = RotaryEmbedding(dim=self.config.hidden_size // self.config.num_heads,
                                              max_position_embeddings=self.config.seq_len + self.puzzle_emb_len,
                                              base=self.config.rope_theta)
        elif self.config.pos_encodings == "learned":
            self.embed_pos = CastedEmbedding(self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        else:
            pass

        # Reasoning Layers
        self.L_level = TinyRecursiveReasoningModel_ACTV1ReasoningModule(layers=[TinyRecursiveReasoningModel_ACTV1Block(self.config) for _i in range(self.config.L_layers)])

        # Initial states
        self.L_init = nn.Buffer(trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)

        # Q head special init
        # Init Q to (almost) zero for faster learning during bootstrapping
        with torch.no_grad():
            self.q_head.weight.zero_()
            self.q_head.bias.fill_(-5)  # type: ignore

    def _input_embeddings(self, input: torch.Tensor, puzzle_identifiers: torch.Tensor):
        # Token embedding
        embedding = self.embed_tokens(input.to(torch.int32))

        # Puzzle embeddings
        if self.config.puzzle_emb_ndim > 0:
            puzzle_embedding = self.puzzle_emb(puzzle_identifiers)
            
            pad_count = self.puzzle_emb_len * self.config.hidden_size - puzzle_embedding.shape[-1]
            if pad_count > 0:
                puzzle_embedding = F.pad(puzzle_embedding, (0, pad_count))

            embedding = torch.cat((puzzle_embedding.view(-1, self.puzzle_emb_len, self.config.hidden_size), embedding), dim=-2)

        # Position embeddings
        if self.config.pos_encodings == "learned":
            # scale by 1/sqrt(2) to maintain forward variance
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight.to(self.forward_dtype))

        # Scale
        return self.embed_scale * embedding

    def empty_carry(self, batch_size: int, seq_len: int = None, device: torch.device = None):
        if seq_len is None:
            seq_len = self.config.seq_len
        return TinyRecursiveReasoningModel_ACTV1InnerCarry(
            z_L=torch.empty(batch_size, seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype, device=device),
        )
        
    def reset_carry(self, reset_flag: torch.Tensor, carry: TinyRecursiveReasoningModel_ACTV1InnerCarry):
        return TinyRecursiveReasoningModel_ACTV1InnerCarry(
            z_L=torch.where(reset_flag.view(-1, 1, 1), self.L_init, carry.z_L),
        )

    def forward(self, carry: TinyRecursiveReasoningModel_ACTV1InnerCarry, batch: Dict[str, torch.Tensor]) -> Tuple[TinyRecursiveReasoningModel_ACTV1InnerCarry, torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        seq_info = dict(
            cos_sin=self.rotary_emb() if hasattr(self, "rotary_emb") else None,
        )

        # Input encoding
        input_embeddings = self._input_embeddings(batch["inputs"], batch["puzzle_identifiers"])

        # Slice RoPE to match actual sequence length (supports dynamic-length generation)
        if seq_info["cos_sin"] is not None:
            actual_len = input_embeddings.shape[1]
            cos, sin = seq_info["cos_sin"]
            seq_info["cos_sin"] = (cos[:actual_len], sin[:actual_len])

        # Forward iterations
        it = 0
        z_L = carry.z_L
        # H_cycles-1 without grad
        with torch.no_grad():
            for _H_step in range(self.config.H_cycles-1):
                for _L_step in range(self.config.L_cycles):
                    z_L = self.L_level(z_L + input_embeddings, **seq_info)
                z_L = self.L_level(z_L, **seq_info)
        # 1 with grad
        for _L_step in range(self.config.L_cycles):
            z_L = self.L_level(z_L + input_embeddings, **seq_info)
        z_L = self.L_level(z_L, **seq_info)
        z_out = z_L

        # LM Outputs
        new_carry = TinyRecursiveReasoningModel_ACTV1InnerCarry(z_L=z_L.detach())  # New carry no grad
        output = self.lm_head(z_out)[:, self.puzzle_emb_len:]
        q_logits = self.q_head(z_out[:, 0]).to(torch.float32) # Q-head; uses the first puzzle_emb position
        return new_carry, output, (q_logits[..., 0], q_logits[..., 1])


class TinyRecursiveReasoningModel_ACTV1(nn.Module):
    """ACT wrapper."""

    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = TinyRecursiveReasoningModel_ACTV1Config(**config_dict)
        self.inner = TinyRecursiveReasoningModel_ACTV1_Inner(self.config)

    @property
    def puzzle_emb(self):
        return self.inner.puzzle_emb

    def initial_carry(self, batch: Dict[str, torch.Tensor]):
        batch_size = batch["inputs"].shape[0]
        seq_len = batch["inputs"].shape[1]
        device = batch["inputs"].device

        return TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=self.inner.empty_carry(batch_size, seq_len, device),

            steps=torch.zeros((batch_size, ), dtype=torch.int32, device=device),
            halted=torch.ones((batch_size, ), dtype=torch.bool, device=device),  # Default to halted
            
            current_data={k: torch.empty_like(v) for k, v in batch.items()}
        )
        
    def forward(self, carry: TinyRecursiveReasoningModel_ACTV1Carry, batch: Dict[str, torch.Tensor]) -> Tuple[TinyRecursiveReasoningModel_ACTV1Carry, Dict[str, torch.Tensor]]:

        # Update data, carry (removing halted sequences)
        new_inner_carry = self.inner.reset_carry(carry.halted, carry.inner_carry)
        
        new_steps = torch.where(carry.halted, 0, carry.steps)

        new_current_data = {k: torch.where(carry.halted.view((-1, ) + (1, ) * (batch[k].ndim - 1)), batch[k], v) for k, v in carry.current_data.items()}

        # Forward inner model
        new_inner_carry, logits, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)

        outputs = {
            "logits": logits,
            "q_halt_logits": q_halt_logits,
            "q_continue_logits": q_continue_logits
        }

        with torch.no_grad():
            # Step
            new_steps = new_steps + 1
            is_last_step = new_steps >= self.config.halt_max_steps
            
            halted = is_last_step

            # if training, and ACT is enabled
            if self.training and (self.config.halt_max_steps > 1):

                # Halt signal
                # NOTE: During evaluation, always use max steps, this is to guarantee the same halting steps inside a batch for batching purposes
                
                if self.config.no_ACT_continue:
                    halted = halted | (q_halt_logits > 0)
                else:
                    halted = halted | (q_halt_logits > q_continue_logits)

                # Exploration
                min_halt_steps = (torch.rand_like(q_halt_logits) < self.config.halt_exploration_prob) * torch.randint_like(new_steps, low=2, high=self.config.halt_max_steps + 1)
                halted = halted & (new_steps >= min_halt_steps)

                if not self.config.no_ACT_continue:
                    # Compute target Q
                    # NOTE: No replay buffer and target networks for computing target Q-value.
                    # As batch_size is large, there're many parallel envs.
                    # Similar concept as PQN https://arxiv.org/abs/2407.04811
                    _, _, (next_q_halt_logits, next_q_continue_logits), _, _ = self.inner(new_inner_carry, new_current_data)
                    outputs["target_q_continue"] = torch.sigmoid(torch.where(is_last_step, next_q_halt_logits, torch.maximum(next_q_halt_logits, next_q_continue_logits)))

        return TinyRecursiveReasoningModel_ACTV1Carry(new_inner_carry, new_steps, halted, new_current_data), outputs

    @torch.no_grad()
    def generate(self, input_ids: torch.Tensor, max_new_tokens: int, temperature: float = 0.1, repetition_penalty: float = 1.2):
        self.eval()
        b_size = input_ids.shape[0]
        
        # We need to know the EOS token ID to stop properly
        # Usually 3 in HF Tokenizers, but you should verify your specific ID!
        eos_token_id = tokenizer.token_to_id("[EOS]")
        
        for step in range(max_new_tokens):
            batch = {
                "inputs": input_ids,
                "puzzle_identifiers": torch.zeros((b_size, self.config.num_puzzle_identifiers), device=input_ids.device, dtype=torch.int32)
            }
            carry = self.initial_carry(batch)
            
            for _ in range(self.config.halt_max_steps):
                carry, outputs = self.forward(carry, batch)
                if carry.halted.all():
                    break
            
            next_token_logits = outputs["logits"][:, -1, :].clone()
            
            # --- NEW: REPETITION PENALTY ---
            # Penalize tokens that have already been generated in the sequence
            for i in range(b_size):
                for token_id in set(input_ids[i].tolist()):
                    if next_token_logits[i, token_id] < 0:
                        next_token_logits[i, token_id] *= repetition_penalty
                    else:
                        next_token_logits[i, token_id] /= repetition_penalty
            
            # --- NEW: PREVENT IMMEDIATE EOS PANIC ---
            # Forbid the model from outputting EOS as the very first generated token
            if step == 0:
                next_token_logits[:, eos_token_id] = float('-inf')

            next_token_logits = next_token_logits / temperature
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # --- THE BRAKES ---
            if next_token.item() == eos_token_id: 
                break
            
            input_ids = torch.cat([input_ids, next_token], dim=1)
            
        return input_ids


ModuleNotFoundError: No module named 'models'

In [31]:
config_dict = {
    # 1. Sequence & Vocabulary (The Generative Shift)
    "batch_size": 16,               
    "seq_len": 512,                 # Maximum context window for text
    "vocab_size": 4096,             # Must match your Micro-BPE tokenizer!
    
    # 2. Transformer Core (~7M Params)
    "hidden_size": 512,             
    "expansion": 4.0,               
    "num_heads": 8,                 
    "pos_encodings": "rope",        # CRITICAL: Replaces 'none' from Sudoku
    
    # 3. Recursion & Loops
    "L_layers": 2,                  # Physical layers
    "H_layers": 1,                  # (Ignored)
    "L_cycles": 6,                  # Inner loops per cycle
    "H_cycles": 1,                 # Deep recursive "thinking" steps
    
    # 4. Adaptive Computation Time (ACT)
    "halt_max_steps": 16,           
    "halt_exploration_prob": 0.1,   
    
    # 5. Stability & Normalization
    "rms_norm_eps": 1e-5,
    "rope_theta": 10000.0,
    "forward_dtype": "bfloat16",    
    
    # 6. Puzzle Disablement (Turning off Sudoku features)
    "mlp_t": False,                 # CRITICAL: Must be False to use Attention
    "puzzle_emb_ndim": 0,           # Turn off puzzle embeddings
    "num_puzzle_identifiers": 1,    # Safe dummy value
    "puzzle_emb_len": 0,
    "no_ACT_continue": True, 
    "causal": True    
}

In [33]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import math
from upload import TinyRecursiveReasoningModel_ACTV1
# Assume your model and loaders are imported/defined above
# from trm_singlez import TinyRecursiveReasoningModel_ACTV1, TinyRecursiveReasoningModel_ACTV1Config
# gsm8k_loader, tinystories_loader = ... 

# ==========================================
# 1. Initialization & VRAM Optimization
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Initialize Model (Make sure config has causal=True and vocab_size=4096)
model = TinyRecursiveReasoningModel_ACTV1(config_dict).to(device)

# Standard optimizer for LLMs
optimizer = AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)

# Mixed Precision Scaler (Crucial for 4GB/8GB GPUs)
scaler = torch.amp.GradScaler('cuda')

# Gradient Accumulation: Simulates a larger batch size without crashing VRAM
accumulation_steps = 4

Training on device: cuda


In [34]:
# ==========================================
# 2. The Core Training Function
# ==========================================
def train_epoch(model, dataloader, optimizer, scaler, epoch, phase_name):
    model.train()
    total_loss = 0
    
    for batch_idx, batch in enumerate(dataloader):
        # Move tensors to GPU
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        
        # Prepare the custom carry state for your TRM
        trm_batch = {
            "inputs": input_ids,
            "puzzle_identifiers": torch.zeros((input_ids.shape[0], model.config.num_puzzle_identifiers), dtype=torch.int32, device=device)
        }
        carry = model.initial_carry(trm_batch)
        
        # Mixed Precision Forward Pass
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            # Your TRM forward pass
            _, outputs = model(carry, trm_batch)
            logits = outputs["logits"]
            
            # --- NEXT TOKEN PREDICTION SHIFT ---
            # # Model predicts token t+1 based on tokens 0 to t
            # shift_logits = logits[..., :-1, :].contiguous()
            # shift_labels = labels[..., 1:].contiguous()
            
            # Flatten to calculate Cross Entropy
            # -100 in shift_labels will be automatically ignored!
            loss = F.cross_entropy(
                logits.view(-1, model.config.vocab_size), 
                labels.view(-1)
            )
            
            # Scale loss for gradient accumulation
            loss = loss / accumulation_steps

        # Backward Pass with Scaler
        scaler.scale(loss).backward()
        
        # Step Optimizer every `accumulation_steps`
        if (batch_idx + 1) % accumulation_steps == 0:
            # Gradient clipping to prevent exploding gradients in recursive loops
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulation_steps
        
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch} [{phase_name}] | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item() * accumulation_steps:.4f}")

    return total_loss / len(dataloader)

In [ ]:
# ==========================================
# 3. Execution (The Curriculum)
# ==========================================
num_tinystories_epochs = 2
# num_gsm8k_epochs = 5

print("\n=== PHASE 1: Syntax (TinyStories) ===")
for epoch in range(1, num_tinystories_epochs + 1):
    avg_loss = train_epoch(model, tinystories_loader, optimizer, scaler, epoch, "TinyStories")
    print(f"--> Phase 1, Epoch {epoch} Complete. Avg Loss: {avg_loss:.4f}\n")

    # NEW: Save Phase 1 weights!
    torch.save(model.state_dict(), f"g_trm_tinystories_epoch_{epoch}.pt")

# print("\n=== PHASE 2: Reasoning (GSM8K) ===")
# # You might want to lower the learning rate for the complex reasoning phase
# for param_group in optimizer.param_groups:
#     param_group['lr'] = 1e-4

# for epoch in range(1, num_gsm8k_epochs + 1):
#     avg_loss = train_epoch(model, gsm8k_loader, optimizer, scaler, epoch, "GSM8K")
#     print(f"--> Phase 2, Epoch {epoch} Complete. Avg Loss: {avg_loss:.4f}\n")
    
#     # Save checkpoint
#     torch.save(model.state_dict(), f"g_trm_gsm8k_epoch_{epoch}.pt")

# print("Training Complete!")


=== PHASE 1: Syntax (TinyStories) ===
Epoch 1 [TinyStories] | Batch 0/6250 | Loss: 8.8257
Epoch 1 [TinyStories] | Batch 50/6250 | Loss: 6.0671
Epoch 1 [TinyStories] | Batch 100/6250 | Loss: 6.0026
Epoch 1 [TinyStories] | Batch 150/6250 | Loss: 6.0275
Epoch 1 [TinyStories] | Batch 200/6250 | Loss: 5.7607
Epoch 1 [TinyStories] | Batch 250/6250 | Loss: 5.8474
Epoch 1 [TinyStories] | Batch 300/6250 | Loss: 5.6147
Epoch 1 [TinyStories] | Batch 350/6250 | Loss: 5.0871
Epoch 1 [TinyStories] | Batch 400/6250 | Loss: 4.9354
Epoch 1 [TinyStories] | Batch 450/6250 | Loss: 4.2004
Epoch 1 [TinyStories] | Batch 500/6250 | Loss: 4.1095
Epoch 1 [TinyStories] | Batch 550/6250 | Loss: 3.8262
Epoch 1 [TinyStories] | Batch 600/6250 | Loss: 3.6440
Epoch 1 [TinyStories] | Batch 650/6250 | Loss: 3.5216
Epoch 1 [TinyStories] | Batch 700/6250 | Loss: 3.6773
Epoch 1 [TinyStories] | Batch 750/6250 | Loss: 3.4355


In [16]:
num_params = sum(p.numel() for p in model.parameters())
size = num_params * 4 / (1024 ** 2)
print(f"Model size: {size:.2f} MB")

Model size: 14.50 MB


In [8]:
import torch
import re
from datasets import load_dataset
from tokenizers import Tokenizer
# from trm_singlez import TinyRecursiveReasoningModel_ACTV1

# ==========================================
# 1. Initialization
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Initializing Evaluation Pipeline...")

# Load your custom tokenizer
tokenizer = Tokenizer.from_file("./trm_tokenizer/tokenizer.json")

# Load model (Use the exact config_dict from your training script)
model = TinyRecursiveReasoningModel_ACTV1(config_dict)
model.load_state_dict(torch.load("g_trm_gsm8k_epoch_5.pt", map_location=device))
model.to(device)
model.eval()

# Load the GSM8K Test Split (Not the train split!)
test_dataset = load_dataset("gsm8k", "main", split="test")

# ==========================================
# 2. Answer Extraction Logic
# ==========================================
def extract_model_answer(generated_text: str):
    """Uses Regex to find the number inside the [ANSWER] tags."""
    match = re.search(r'\[ANSWER\](.*?)\[/ANSWER\]', generated_text, re.DOTALL)
    if match:
        numbers = re.findall(r'-?\d+\.?\d*', match.group(1))
        return float(numbers[-1]) if numbers else None
    return None

def extract_ground_truth(raw_answer: str):
    """Extracts the final number after #### in the GSM8K dataset."""
    parts = raw_answer.split("####")
    if len(parts) > 1:
        numbers = re.findall(r'-?\d+\.?\d*', parts[1])
        return float(numbers[-1]) if numbers else None
    return None

# ==========================================
# 3. The Evaluation Loop
# ==========================================
total_questions = 0
correct_answers = 0

# Dictionaries to track the "Inference Gap"
difficulty_tracker = {"Easy (1-2 steps)": [0, 0], "Medium (3-4 steps)": [0, 0], "Hard (5+ steps)": [0, 0]}

print(f"Starting evaluation on {len(test_dataset)} test questions...\n")

for i, example in enumerate(test_dataset):
    question = example["question"]
    ground_truth_text = example["answer"]
    
    num_steps = ground_truth_text.split("####")[0].count('\n')
    if num_steps <= 2:
        difficulty_category = "Easy (1-2 steps)"
    elif num_steps <= 4:
        difficulty_category = "Medium (3-4 steps)"
    else:
        difficulty_category = "Hard (5+ steps)"

    prompt = f"[INST] {question} [/INST]\n[THINK]"
    input_ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long).to(device)

    # Generate with strict temperature and NO repetition penalty
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=300, temperature=0.1)
    
    generated_text = tokenizer.decode(output_ids[0].tolist(), skip_special_tokens=False)
    
    model_ans = extract_model_answer(generated_text)
    truth_ans = extract_ground_truth(ground_truth_text)

    # ==========================================
    # INDENTATION FIXED BELOW
    # ==========================================
    total_questions += 1
    difficulty_tracker[difficulty_category][1] += 1
    
    is_correct = False
    if model_ans is not None and truth_ans is not None:
        if model_ans == truth_ans:
            correct_answers += 1
            difficulty_tracker[difficulty_category][0] += 1
            is_correct = True

    # if i % 10 == 0 or model_ans is None: 
    print(f"\n[{i}/{len(test_dataset)}] Accuracy so far: {(correct_answers/total_questions)*100:.2f}%")
    if not is_correct:
        print(f"  -> Expected: {truth_ans}")
        print(f"  -> Extracted: {model_ans}")
        print(f"  -> RAW MODEL OUTPUT:\n{generated_text}\n")
        print("-" * 40)

# ==========================================
# 4. Final Thesis Metrics Output
# ==========================================
final_accuracy = (correct_answers / total_questions) * 100
print("\n" + "="*50)
print(f"FINAL MODEL ACCURACY: {final_accuracy:.2f}%")
print("="*50)
print("INFERENCE GAP ANALYSIS (For Presentation Graph):")
for category, counts in difficulty_tracker.items():
    correct, total = counts
    if total > 0:
        acc = (correct / total) * 100
        print(f"- {category}: {acc:.2f}% ({correct}/{total})")

Initializing Evaluation Pipeline...
Starting evaluation on 1319 test questions...


[0/1319] Accuracy so far: 0.00%
  -> Expected: 18.0
  -> Extracted: 0.0
  -> RAW MODEL OUTPUT:
[BOS] [INST] J an et ’ s ducks lay 1 6 eggs per day . She eat s three for breakfast every morning and bake s muffin s for her friends every day with four . She sell s the re ma ind er at the farm ers ' market da ily for $ 2 per fresh duck egg . How much in doll ars does she make every day at the farm ers ' market ? [/INST] [THINK] [EOS] 3 + 4 = 5 days to cook a week . The total number of eggs that are left is $ 8 x $ 0 = $ 7 0 . So , the total number of eggs in the field was $ 9 0 - $ 7 0 = $ 1 0 . [/THINK] [ANSWER] 1 0 [/ANSWER]

----------------------------------------

[1/1319] Accuracy so far: 0.00%
  -> Expected: 3.0
  -> Extracted: 5.0
  -> RAW MODEL OUTPUT:
[BOS] [INST] A robe takes 2 bol ts of blue fi ber and half that much white fi ber . How many bol ts in total does it take ? [/INST] [THINK] [EOS] 1 

KeyboardInterrupt: 

Synthetic Data

In [9]:
import json
import random

# Configuration
NUM_SAMPLES = 50000
OUTPUT_FILE = "phase_1_5_synthetic_math.jsonl"

def generate_math_problem():
    # 1. Pick an operation (Weighted towards addition/subtraction for basic logic)
    op = random.choices(['+', '-', '*'], weights=[0.4, 0.4, 0.2])[0]
    
    # 2. Generate numbers (Curriculum bounds: keep numbers small so the 7M model learns the *algorithm*, not just memorizes noise)
    if op == '*':
        # Multiplication: 1 to 20
        num1 = random.randint(1, 20)
        num2 = random.randint(1, 20)
        ans = num1 * num2
        op_word = "multiplication"
    elif op == '+':
        # Addition: 1 to 100
        num1 = random.randint(1, 100)
        num2 = random.randint(1, 100)
        ans = num1 + num2
        op_word = "addition"
    else:
        # Subtraction: 1 to 100 (Ensure positive answers to keep it simple for now)
        num1 = random.randint(1, 100)
        num2 = random.randint(1, num1) 
        ans = num1 - num2
        op_word = "subtraction"

    # 3. Construct the text perfectly matching your GSM8K template
    question = f"Evaluate: {num1} {op} {num2}"
    
    # Notice the explicit, deterministic Chain-of-Thought
    thought = f"The operation is {op_word}. {num1} {op} {num2} = {ans}."
    
    full_text = f"[INST] {question} [/INST]\n[THINK]\n{thought}\n[/THINK]\n[ANSWER] {ans} [/ANSWER]"
    
    return {
        "question": question,
        "answer": f"{thought} #### {ans}", # Keep the GSM8K extraction format just in case
        "text": full_text
    }

print(f"Generating {NUM_SAMPLES} synthetic math problems...")

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for _ in range(NUM_SAMPLES):
        data = generate_math_problem()
        f.write(json.dumps(data) + '\n')

print(f"Done! Saved to {OUTPUT_FILE}")

Generating 50000 synthetic math problems...
Done! Saved to phase_1_5_synthetic_math.jsonl


In [17]:
from datasets import load_dataset

# Load Phase 1.5 Data
synthetic_math_dataset = load_dataset("json", data_files="phase_1_5_synthetic_math.jsonl", split="train")

# You can now pass this dataset into your exact same tokenize_and_prepare_labels function!

Generating train split: 50000 examples [00:00, 802482.66 examples/s]


In [19]:
# ==========================================
# 4. Pipeline Execution
# ==========================================
print("Loading Datasets...")
# Load GSM8K
gsm8k_dataset = load_dataset("gsm8k", "main", split="train")

# Load a subset of TinyStories (e.g., 100,000 stories so it doesn't overwhelm math)
tinystories_dataset = load_dataset("roneneldan/TinyStories", split="train[:100000]")

synthetic_math_dataset = load_dataset("json", data_files="phase_1_5_synthetic_math.jsonl", split="train")



Loading Datasets...


In [23]:
print(gsm8k_dataset[0]) 
print(tinystories_dataset[0]) 
print(synthetic_math_dataset[0])

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they h

In [ ]:
print("Formatting Text...")
# Apply the formatting templates
gsm8k_dataset = gsm8k_dataset.map(format_gsm8k, remove_columns=gsm8k_dataset.column_names)
tinystories_dataset = tinystories_dataset.map(format_tinystories, remove_columns=tinystories_dataset.column_names)



Formatting Text...


In [28]:
print(gsm8k_dataset[0])
print(synthetic_math_dataset[0])

{'text': '[INST] Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? [/INST]\n[THINK]\nNatalia sold 48/2 = 24 clips in May.\nNatalia sold 48+24 = 72 clips altogether in April and May.\n[/THINK]\n[ANSWER] 72 [/ANSWER]'}
{'question': 'Evaluate: 85 + 20', 'answer': 'The operation is addition. 85 + 20 = 105. #### 105', 'text': '[INST] Evaluate: 85 + 20 [/INST]\n[THINK]\nThe operation is addition. 85 + 20 = 105.\n[/THINK]\n[ANSWER] 105 [/ANSWER]'}


In [ ]:
print("Tokenizing... (This is very fast)")
# Apply tokenization using batched mapping
gsm8k_tokenized = gsm8k_dataset.map(tokenize_and_prepare_labels, batched=True, batch_size=1000, remove_columns=["text"])
tinystories_tokenized = tinystories_dataset.map(tokenize_and_prepare_labels, batched=True, batch_size=1000, remove_columns=["text"])

# Tell HF datasets to return actual PyTorch Tensors
gsm8k_tokenized.set_format(type="torch", columns=["input_ids", "labels"])
tinystories_tokenized.set_format(type="torch", columns=["input_ids", "labels"])

Tokenizing... (This is very fast)


NameError: name 'tokenize_and_prepare_labels' is not defined

Only look at inst an answer for gradient

In [51]:
def tokenize_and_prepare_labels(example):
    # Tokenize the entire text
    encoded = tokenizer.encode(example['text'])
    input_ids = encoded.ids
    
    # --- THE FIX: STRICT TRUNCATION ---
    MAX_SEQ_LEN = 512  # Must match your model's config.seq_len
    input_ids = input_ids[:MAX_SEQ_LEN]
    
    # Clone to create labels
    labels = input_ids.copy()
    
    # Find exactly where the prompt ends
    prompt_end_str = "[/INST]"
    prompt_end_idx = example['text'].find(prompt_end_str)
    
    if prompt_end_idx != -1:
        # Extract just the prompt text
        prompt_text = example['text'][:prompt_end_idx + len(prompt_end_str)]
        prompt_token_count = len(tokenizer.encode(prompt_text).ids)
        
        # Mask out the prompt tokens (Safeguarded against truncation)
        mask_limit = min(prompt_token_count, len(labels))
        for i in range(mask_limit):
            labels[i] = -100
            
    return {
        "input_ids": input_ids,
        "labels": labels
    }

# Apply the mapping
print("Tokenizing and Masking Datasets...")
synthetic_math_tokenized = synthetic_math_dataset.map(tokenize_and_prepare_labels)
gsm8k_tokenized = gsm8k_dataset.map(tokenize_and_prepare_labels)




Tokenizing and Masking Datasets...


Map: 100%|██████████| 7473/7473 [00:02<00:00, 2759.36 examples/s]


In [52]:
gsm8k_tokenized.set_format(type="torch", columns=["input_ids", "labels"])

In [53]:
synthetic_math_tokenized.set_format(type="torch", columns=["input_ids", "labels"])

In [54]:
synthetic_math_tokenized[0]

{'input_ids': tensor([   2,    4,   48, 2686,  483,   37,   35,   32,   22,   29,   27,    5,
            6,  177,  398,  171, 1122,  178, 1600,  169,  593,   25,   35,   32,
           22,   29,   27,   40,   28,   27,   32,   25,    7,    8,   28,   27,
           32,    9,    3]),
 'labels': tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100,  177,  398,  171, 1122,  178, 1600,  169,  593,   25,   35,   32,
           22,   29,   27,   40,   28,   27,   32,   25,    7,    8,   28,   27,
           32,    9,    3])}

In [55]:
from torch.utils.data import DataLoader

gsm8k_loader = DataLoader(gsm8k_tokenized, batch_size=BATCH_SIZE, shuffle=True, 
    collate_fn=trm_collate_fn)
synthetic_math_loader = DataLoader(synthetic_math_tokenized, batch_size=BATCH_SIZE, shuffle=True, 
    collate_fn=trm_collate_fn)

print("\n--- Pipeline Complete! ---")
print(f"GSM8K Batches available: {len(gsm8k_loader)}")
print(f"Synthetic Math Batches available: {len(synthetic_math_loader)}")

# Let's verify a batch!
sample_batch = next(iter(gsm8k_loader))
print(f"\nShape of input_ids: {sample_batch['input_ids'].shape}")
print(f"Shape of labels: {sample_batch['labels'].shape}")
print(f"First 10 input_ids: {sample_batch['input_ids'][0][:10]}")
print(f"Last 10 labels (Notice the -100 padding!): {sample_batch['labels'][0][-10:]}")


--- Pipeline Complete! ---
GSM8K Batches available: 468
Synthetic Math Batches available: 3125

Shape of input_ids: torch.Size([16, 292])
Shape of labels: torch.Size([16, 292])
First 10 input_ids: tensor([  2,   4,  54, 174, 197, 194, 681,  15,  32,  27])
Last 10 labels (Notice the -100 padding!): tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100])


Training Loop

In [56]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import math

# Assume your model and loaders are imported/defined above
# from trm_singlez import TinyRecursiveReasoningModel_ACTV1, TinyRecursiveReasoningModel_ACTV1Config
# gsm8k_loader, tinystories_loader = ... 

# ==========================================
# 1. Initialization & VRAM Optimization
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Initialize Model (Make sure config has causal=True and vocab_size=4096)
model = TinyRecursiveReasoningModel_ACTV1(config_dict).to(device)

# Standard optimizer for LLMs
optimizer = AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)

# Mixed Precision Scaler (Crucial for 4GB/8GB GPUs)
scaler = torch.amp.GradScaler('cuda')

# Gradient Accumulation: Simulates a larger batch size without crashing VRAM
accumulation_steps = 4

Training on device: cuda


In [57]:
# ==========================================
# 2. The Core Training Function (Early Stop Edition)
# ==========================================
def train_epoch(model, dataloader, optimizer, scaler, epoch, phase_name, stop_threshold=None):
    model.train()
    total_loss = 0
    recent_loss_sum = 0 # Track loss for the moving average
    
    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        
        trm_batch = {
            "inputs": input_ids,
            "puzzle_identifiers": torch.zeros((input_ids.shape[0], model.config.num_puzzle_identifiers), dtype=torch.int32, device=device)
        }
        carry = model.initial_carry(trm_batch)
        
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            _, outputs = model(carry, trm_batch)
            logits = outputs["logits"]
            
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            
            loss = F.cross_entropy(
                shift_logits.view(-1, model.config.vocab_size), 
                shift_labels.view(-1)
            )
            
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()
        
        if (batch_idx + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        actual_loss = loss.item() * accumulation_steps
        total_loss += actual_loss
        recent_loss_sum += actual_loss
        
        # --- LOGGING & EARLY STOPPING CHECK ---
        if batch_idx > 0 and batch_idx % 20 == 0:
            # Calculate the moving average over the last 50 batches
            smooth_loss = recent_loss_sum / 50
            print(f"Epoch {epoch} [{phase_name}] | Batch {batch_idx}/{len(dataloader)} | Smooth Loss: {smooth_loss:.4f}")
            
            if stop_threshold is not None and smooth_loss <= stop_threshold:
                print("\n" + "="*50)
                print(f"🚨 EARLY STOPPING TRIGGERED 🚨")
                print(f"Smooth Loss ({smooth_loss:.4f}) hit the target threshold of {stop_threshold}.")
                print("Halting training to preserve Phase 1 syntax weights!")
                print("="*50 + "\n")
                
                # Return the average loss up to this point, and True for early_stop
                return total_loss / (batch_idx + 1), True 
            
            # Reset the moving average tracker
            recent_loss_sum = 0

    # If it finishes the epoch without triggering the stop
    return total_loss / len(dataloader), False

In [58]:
# ==========================================
# 3. Execution (The Curriculum)
# ==========================================
num_synthetic_math_epochs = 1
num_gsm8k_epochs = 5

print("\n=== LOADING PHASE 1 (Syntax) WEIGHTS ===")
model.load_state_dict(torch.load("g_trm_tinystories_epoch_2.pt", map_location=device))
print("TinyStories weights loaded successfully!")

print("\n=== PHASE 1.5: Arithmetic (Synthetic Math) ===")
TARGET_LOSS = 0.15 

for epoch in range(1, num_synthetic_math_epochs + 1):
    # THE FIX: Unpack both avg_loss and early_stop!
    avg_loss, early_stop = train_epoch(model, synthetic_math_loader, optimizer, scaler, epoch, "Synthetic Math", stop_threshold=TARGET_LOSS)
    print(f"--> Phase 1.5, Epoch {epoch} Complete/Halted. Avg Loss: {avg_loss:.4f}\n")

    # Save Phase 1.5 weights immediately
    torch.save(model.state_dict(), f"g_trm_synthetic_math_epoch_{epoch}.pt")
    
    # Break out of the epoch loop if the model learned enough
    if early_stop:
        print("Curriculum goal met. Moving directly to Phase 2...")
        break

print("\n=== PHASE 2: Reasoning (GSM8K) ===")
for param_group in optimizer.param_groups:
    param_group['lr'] = 1e-4

for epoch in range(1, num_gsm8k_epochs + 1):
    # Phase 2 doesn't use early stopping, so we still unpack the second value but ignore it
    avg_loss, _ = train_epoch(model, gsm8k_loader, optimizer, scaler, epoch, "GSM8K_Syn")
    print(f"--> Phase 2, Epoch {epoch} Complete. Avg Loss: {avg_loss:.4f}\n")
    
    torch.save(model.state_dict(), f"g_trm_gsm8k_after_syn_epoch_{epoch}.pt")

print("Training Complete!")


=== LOADING PHASE 1 (Syntax) WEIGHTS ===
TinyStories weights loaded successfully!

=== PHASE 1.5: Arithmetic (Synthetic Math) ===
Epoch 1 [Synthetic Math] | Batch 20/3125 | Smooth Loss: 2.1649
Epoch 1 [Synthetic Math] | Batch 40/3125 | Smooth Loss: 0.8612
Epoch 1 [Synthetic Math] | Batch 60/3125 | Smooth Loss: 0.5689
Epoch 1 [Synthetic Math] | Batch 80/3125 | Smooth Loss: 0.4689
Epoch 1 [Synthetic Math] | Batch 100/3125 | Smooth Loss: 0.4046
Epoch 1 [Synthetic Math] | Batch 120/3125 | Smooth Loss: 0.3603
Epoch 1 [Synthetic Math] | Batch 140/3125 | Smooth Loss: 0.3291
Epoch 1 [Synthetic Math] | Batch 160/3125 | Smooth Loss: 0.2896
Epoch 1 [Synthetic Math] | Batch 180/3125 | Smooth Loss: 0.2476
Epoch 1 [Synthetic Math] | Batch 200/3125 | Smooth Loss: 0.1941
Epoch 1 [Synthetic Math] | Batch 220/3125 | Smooth Loss: 0.1478

🚨 EARLY STOPPING TRIGGERED 🚨
Smooth Loss (0.1478) hit the target threshold of 0.15.
Halting training to preserve Phase 1 syntax weights!

--> Phase 1.5, Epoch 1 Complet

Evaluation

In [61]:
import torch
import re
from datasets import load_dataset
from tokenizers import Tokenizer
# from trm_singlez import TinyRecursiveReasoningModel_ACTV1

# ==========================================
# 1. Initialization
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Initializing Evaluation Pipeline...")

# Load your custom tokenizer
tokenizer = Tokenizer.from_file("./trm_tokenizer/tokenizer.json")

# Load model (Use the exact config_dict from your training script)
model = TinyRecursiveReasoningModel_ACTV1(config_dict)
model.load_state_dict(torch.load("g_trm_gsm8k_epoch_5.pt", map_location=device))
model.to(device)
model.eval()

# Load the GSM8K Test Split (Not the train split!)
test_dataset = load_dataset("gsm8k", "main", split="test")

# ==========================================
# 2. Answer Extraction Logic
# ==========================================
def extract_model_answer(generated_text: str):
    """Uses Regex to find the number inside the [ANSWER] tags."""
    match = re.search(r'\[ANSWER\](.*?)\[/ANSWER\]', generated_text, re.DOTALL)
    if match:
        numbers = re.findall(r'-?\d+\.?\d*', match.group(1))
        return float(numbers[-1]) if numbers else None
    return None

def extract_ground_truth(raw_answer: str):
    """Extracts the final number after #### in the GSM8K dataset."""
    parts = raw_answer.split("####")
    if len(parts) > 1:
        numbers = re.findall(r'-?\d+\.?\d*', parts[1])
        return float(numbers[-1]) if numbers else None
    return None

# ==========================================
# 3. The Evaluation Loop
# ==========================================
total_questions = 0
correct_answers = 0

# Dictionaries to track the "Inference Gap"
difficulty_tracker = {"Easy (1-2 steps)": [0, 0], "Medium (3-4 steps)": [0, 0], "Hard (5+ steps)": [0, 0]}

print(f"Starting evaluation on {len(test_dataset)} test questions...\n")

for i, example in enumerate(test_dataset):
    question = example["question"]
    ground_truth_text = example["answer"]
    
    num_steps = ground_truth_text.split("####")[0].count('\n')
    if num_steps <= 2:
        difficulty_category = "Easy (1-2 steps)"
    elif num_steps <= 4:
        difficulty_category = "Medium (3-4 steps)"
    else:
        difficulty_category = "Hard (5+ steps)"

    prompt = f"[INST] {question} [/INST]\n[THINK]\n"
    input_ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long).to(device)

    # Generate with strict temperature and NO repetition penalty
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=300, temperature=0.1)
    
    generated_text = tokenizer.decode(output_ids[0].tolist(), skip_special_tokens=False)
    
    model_ans = extract_model_answer(generated_text)
    truth_ans = extract_ground_truth(ground_truth_text)

    # ==========================================
    # INDENTATION FIXED BELOW
    # ==========================================
    total_questions += 1
    difficulty_tracker[difficulty_category][1] += 1
    
    is_correct = False
    if model_ans is not None and truth_ans is not None:
        if model_ans == truth_ans:
            correct_answers += 1
            difficulty_tracker[difficulty_category][0] += 1
            is_correct = True

    # if i % 10 == 0 or model_ans is None: 
    print(f"\n[{i}/{len(test_dataset)}] Accuracy so far: {(correct_answers/total_questions)*100:.2f}%")
    if not is_correct:
        print(f"  -> Expected: {truth_ans}")
        print(f"  -> Extracted: {model_ans}")
        print(f"  -> RAW MODEL OUTPUT:\n{generated_text}\n")
        print("-" * 40)

# ==========================================
# 4. Final Thesis Metrics Output
# ==========================================
final_accuracy = (correct_answers / total_questions) * 100
print("\n" + "="*50)
print(f"FINAL MODEL ACCURACY: {final_accuracy:.2f}%")
print("="*50)
print("INFERENCE GAP ANALYSIS (For Presentation Graph):")
for category, counts in difficulty_tracker.items():
    correct, total = counts
    if total > 0:
        acc = (correct / total) * 100
        print(f"- {category}: {acc:.2f}% ({correct}/{total})")

Initializing Evaluation Pipeline...
Starting evaluation on 1319 test questions...


[0/1319] Accuracy so far: 0.00%
  -> Expected: 18.0
  -> Extracted: 2.0
  -> RAW MODEL OUTPUT:
[BOS] [INST] J an et ’ s ducks lay 1 6 eggs per day . She eat s three for breakfast every morning and bake s muffin s for her friends every day with four . She sell s the re ma ind er at the farm ers ' market da ily for $ 2 per fresh duck egg . How much in doll ars does she make every day at the farm ers ' market ? [/INST] [THINK] [EOS] 3 * 4 = 5 0 days So , which is 2 / 2 = 8 eggs [/THINK] [ANSWER] 2 [/ANSWER]

----------------------------------------

[1/1319] Accuracy so far: 0.00%
  -> Expected: 3.0
  -> Extracted: 5.0
  -> RAW MODEL OUTPUT:
[BOS] [INST] A robe takes 2 bol ts of blue fi ber and half that much white fi ber . How many bol ts in total does it take ? [/INST] [THINK] [EOS] 1 / 4 = 6 min im um s of fu el . The ent ire pri ce is 3 * 8 = 5 0 min us a day . So , the store will be 5 0 - 5 = 7 5 min 

KeyboardInterrupt: 

In [64]:
params = sum(p.numel() for p in model.parameters())
params

3801602

# Non Auto Regressivce TRM

The original TRM, was for fixed length task. While recursing the latent vector, it was able to have a global view of the problem at hand, and can tweak its reasoning accordingly. When training in AR mode, future tokens are masked, which disallows a global view of the problem, leading to poor training. Hence, we are moving to NAR methodology, which is very similar to the original paper.
